In [8]:
%pip install torch_geometric ogb gensim

You should consider upgrading via the '/home/hice1/rdurai6/scratch/project/graph-semantic-research-main/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import gzip
import shutil
import numpy as np
import pandas as pd
import torch
import urllib.request
from ogb.nodeproppred import PygNodePropPredDataset
from gensim.models.fasttext import load_facebook_model

# Fix PyTorch compatibility issues with OGB.PygNodePropPredDataset
if not hasattr(torch, '_is_patched_for_ogb'):
    _orig_load = torch.load
    def patched_load(*args, **kwargs):
        if 'weights_only' not in kwargs:
            kwargs['weights_only'] = False
        return _orig_load(*args, **kwargs)
    torch.load = patched_load
    torch._is_patched_for_ogb = True
    print("Applied PyTorch compatibility fix")

In [ ]:
print("Loading OGB graph structure...")
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='../data/')
data = dataset[0]

# Dataset - https://ogb.stanford.edu/docs/nodeprop/#ogbn-arxiv
# Download raw text
raw_text_url = "https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz"
print(f"Downloading raw text...")
raw_df = pd.read_csv(raw_text_url, sep='\t', compression='gzip', names=['paper id', 'title', 'abstract'], low_memory=False)
print(raw_df.shape)

# Load the local mapping to align graph nodes with text
local_mapping_path = "../data/ogbn_arxiv/mapping/nodeidx2paperid.csv.gz"
mapping_df = pd.read_csv(local_mapping_path, compression='gzip', names=['node index', 'paper id'])
print(mapping_df.shape)

# Merge to ensure the text matches node indices 0 to N
ordered_df = mapping_df.merge(raw_df, on='paper id', how='left')
ordered_df['full_text'] = ordered_df['title'].fillna('') + ". " + ordered_df['abstract'].fillna('')
print(ordered_df.shape)

print(f"Example Raw Text (Node 5): {ordered_df['full_text'].iloc[5][:100]}...")


Loading OGB graph structure...


(179720, 3)
(169344, 2)
(169344, 5)
Example Raw Text (Node 5): analysis of asymptotically optimal sampling based motion planning algorithms for lipschitz continuou...


In [11]:

model_dir = '../models/'
os.makedirs(model_dir, exist_ok=True)
FASTTEXT_URL = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz'
FASTTEXT_BIN = os.path.join(model_dir, 'cc.en.300.bin')

if not os.path.exists(FASTTEXT_BIN):
    gz_path = FASTTEXT_BIN + '.gz'
    print(f'Downloading FastText model (7GB)...')
    urllib.request.urlretrieve(FASTTEXT_URL, gz_path)
    print('Decompressing...')
    with gzip.open(gz_path, 'rb') as f_in, open(FASTTEXT_BIN, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(gz_path)

print('Loading FastText model into memory...')
ft_model = load_facebook_model(FASTTEXT_BIN)



Loading FastText model into memory...


In [12]:
# Regex similar to 02_generate_features_fasttext notebook
REGEX = re.compile(r'[^a-z0-9\s]')

def get_fasttext_embedding(text):
    # Cleaning and tokenization
    text = text.lower()
    tokens = REGEX.sub(' ', text).split()

    if not tokens:
        return np.zeros(300)

    vectors = [ft_model.wv[w] for w in tokens]
    return np.mean(vectors, axis=0)

print(f"Generating embeddings for {len(ordered_df)} nodes")

# Actual embedding (169k nodes)
node_features = np.stack([get_fasttext_embedding(t) for t in ordered_df['full_text'].values])



Generating embeddings for 169344 nodes


In [13]:
# Generate .NPZ file
output_path = '../data/arxiv_fasttext.npz'
split_idx = dataset.get_idx_split()
num_nodes = data.num_nodes

# Since ArXiv has 1 fixed split, we repeat it 10 times to match the pipeline.
train_mask = np.zeros(num_nodes, dtype=bool)
train_mask[split_idx['train']] = True
val_mask = np.zeros(num_nodes, dtype=bool)
val_mask[split_idx['valid']] = True
test_mask = np.zeros(num_nodes, dtype=bool)
test_mask[split_idx['test']] = True

np.savez(output_path,
         node_features=node_features.astype(np.float32),
         node_labels=data.y.numpy().flatten(),
         edges=data.edge_index.numpy().T,
         train_masks=np.tile(train_mask, (10, 1)),
         val_masks=np.tile(val_mask, (10, 1)),
         test_masks=np.tile(test_mask, (10, 1))
)

print(f"Successfully generated {output_path}")



Successfully generated ../data/arxiv_fasttext.npz


In [14]:
check = np.load(output_path, allow_pickle=True)
for key in ['node_features', 'node_labels', 'edges', 'train_masks']:
    print(f"{key} shape: {check[key].shape}")

node_features shape: (169344, 300)
node_labels shape: (169343,)
edges shape: (1166243, 2)
train_masks shape: (10, 169343)
